# Deep Q-Network (DQN)

https://docs.pytorch.org/rl/stable/tutorials/getting-started-5.html

In [1]:
import torch
from tensordict.nn import TensorDictModule, TensorDictSequential
from torch import nn
from torchrl.collectors import Collector
from torchrl.data import LazyTensorStorage, ReplayBuffer
from torchrl.envs import GymEnv, StepCounter, TransformedEnv
from torchrl.modules import EGreedyModule, QValueActor
from torchrl.objectives import DQNLoss, SoftUpdate, ValueEstimators
from torchrl import logger

/usr/local/Caskroom/miniforge/base/envs/trl/lib/python3.13/site-packages/torchrl/modules/mcts/scores.py:574: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in enum.member() if you want to preserve the old behavior
  PUCT = functools.partial(PUCTScore, c=5)  # AlphaGo default value
/usr/local/Caskroom/miniforge/base/envs/trl/lib/python3.13/site-packages/torchrl/modules/mcts/scores.py:575: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in enum.member() if you want to preserve the old behavior
  UCB = functools.partial(UCBScore, c=math.sqrt(2))  # default from Auer et al. 2002
/usr/local/Caskroom/miniforge/base/envs/trl/lib/python3.13/site-packages/torchrl/modules/mcts/scores.py:576: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in enum.member() if you want to preserve the old behavior
  UCB1_TUNED = functools.partial(
/usr/local/Caskroom/mini

DQN requires a discrete action space so we use the `CartPole` environment;
it has two actions that are one-hot encoded by the `GymEnv` wrapper.

In [2]:
env = TransformedEnv(GymEnv("CartPole-v1"), StepCounter())

In [3]:
_ = env.set_seed(0)
_ = torch.manual_seed(0)

The value network estimates the action-value `Q(s,a)` for pairs `(s,a)` of observation and action.

It takes an observation as input and outputs a vector of action-values (one per action).
The deterministic policy then selects the action with the highest action-value (via argmax).

In [4]:
value_net = nn.Sequential(
    nn.LazyLinear(64),
    nn.Tanh(),
    nn.LazyLinear(64),
    nn.Tanh(),
    nn.LazyLinear(env.action_spec.shape[-1]),
)

# wrap value network to work with tensordict as input/output
value_module = TensorDictModule(
    value_net, in_keys=["observation"], out_keys=["action_value"]
)

policy_module = QValueActor(value_module, spec=env.action_spec)

In [5]:
_ = env.rollout(10, policy_module)  # initialize lazy layers

In [6]:
LEARNING_RATE = 1e-4

# module to compute the loss: L = Q(s,a) - (r + gamma * max_a Q(s',a))
loss_module = DQNLoss(value_network=policy_module, action_space=env.action_spec)
loss_module.make_value_estimator(ValueEstimators.TD0, gamma=0.99)  # default value

updater = SoftUpdate(loss_module, eps=0.99)  # soft-updater for target weights

optimizer = torch.optim.Adam(loss_module.parameters(), lr=LEARNING_RATE)

Use an epsilon-greedy exploration module to ensure a certain amount of exploration early in the training process, 
since deterministic policies can otherwise get "stuck" due to a lack of variety in the experiences seen.

In [7]:
exploration_module = EGreedyModule(
    spec=env.action_spec,
    eps_init=0.5,
    annealing_num_steps=100_000,
)

policy_explore = TensorDictSequential([policy_module, exploration_module])

The collector takes random actions for a fixed number of steps (on top of using the exploration policy) 
to ensure that the data is varied enough.

Use a large replay buffer for off-policy algorithms since the policy used to obtain experiences is irrelevant;
the Bellman equation should be satisfied for all transitions.

In [8]:
INIT_RANDOM_FRAMES = 5000
FRAMES_PER_BATCH = 100

collector = Collector(
    env,
    policy=policy_explore,
    frames_per_batch=FRAMES_PER_BATCH,
    init_random_frames=INIT_RANDOM_FRAMES,
    total_frames=-1,
)

buffer = ReplayBuffer(storage=LazyTensorStorage(100_000))

At each traninig step, we sample data (using the collector) which is stored in the replay buffer.
Data is then sampled from the buffer and used to train the Q-network (for several iterations, with new samples each time).

We evaluate the policy by looking at the longest trajectory in the latest batch of data 
(since the goal of `CartPole` is to keep the pole upright for as many steps as possible).
We define success as having attained an episode of at least 200 steps.

In [9]:
OPTIM_STEPS = 10
BATCH_SIZE = 128

step_count = 0
episode_count = 0


for idx, data in enumerate(collector):
    buffer.extend(data)  # add data to replay buffer

    step_count += data.numel()
    episode_count += data["next", "done"].sum()

    if len(buffer) < collector.init_random_frames:
        continue

    # length of longest trajectory in batch
    max_steps = data["next", "step_count"].max()

    # define the stopping condition as reaching 200 steps
    if max_steps > 200:
        break

    if idx % 10 == 0:
        logger.info(f"[{idx:>3}] max steps: {max_steps:>3}")

    for _ in range(OPTIM_STEPS):
        sample = buffer.sample(BATCH_SIZE)

        loss_module(sample)["loss"].backward()

        optimizer.step()
        optimizer.zero_grad()

        updater.step()  # update target params

    exploration_module.step(data.numel())  # update exploration factor

logger.info(f"solved after {step_count} steps, {episode_count} episodes")

2026-02-16 11:14:27,251 [torchrl][INFO]    Initialized LazyTensorStorage with torch.Size([100000]) shape [END]
2026-02-16 11:14:31,607 [torchrl][INFO]    [ 50] max steps:  24 [END]
2026-02-16 11:14:32,937 [torchrl][INFO]    [ 60] max steps:  17 [END]
2026-02-16 11:14:34,311 [torchrl][INFO]    [ 70] max steps:  16 [END]
2026-02-16 11:14:35,681 [torchrl][INFO]    [ 80] max steps:  21 [END]
2026-02-16 11:14:37,045 [torchrl][INFO]    [ 90] max steps:  33 [END]
2026-02-16 11:14:38,416 [torchrl][INFO]    [100] max steps:  26 [END]
2026-02-16 11:14:39,794 [torchrl][INFO]    [110] max steps:  28 [END]
2026-02-16 11:14:41,289 [torchrl][INFO]    [120] max steps:  84 [END]
2026-02-16 11:14:42,694 [torchrl][INFO]    [130] max steps:  53 [END]
2026-02-16 11:14:44,088 [torchrl][INFO]    [140] max steps:  80 [END]
2026-02-16 11:14:45,500 [torchrl][INFO]    [150] max steps:  29 [END]
2026-02-16 11:14:46,923 [torchrl][INFO]    [160] max steps:  20 [END]
2026-02-16 11:14:48,360 [torchrl][INFO]    [170] 